In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install -q transformers peft datasets accelerate bitsandbytes evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/llm-lab/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/llm-lab/results', exist_ok=True)
print("Drive mounted and folders ready.")

Mounted at /content/drive
Drive mounted and folders ready.


In [4]:
import os
import json
import time
import torch
from dataclasses import dataclass, field
from typing import Optional, List
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
from datasets import load_dataset
from torch.utils.data import Dataset

# ── Config ─────────────────────────────────────────────────────────

@dataclass
class ExperimentConfig:
    name: str
    model: str
    method: str               # "lora", "qlora", "full"
    dataset: str
    task: str
    num_samples: int
    epochs: int
    batch_size: int
    learning_rate: float
    max_length: int
    output_dir: str
    lora_rank: Optional[int] = None
    lora_alpha: Optional[int] = None
    lora_dropout: Optional[float] = None
    target_modules: Optional[List[str]] = None

# ── Prompt formatting ───────────────────────────────────────────────

def format_instruction(sample):
    if sample.get("input", "").strip():
        return (f"### Instruction:\n{sample['instruction']}\n\n"
                f"### Input:\n{sample['input']}\n\n"
                f"### Response:\n{sample['output']}")
    return (f"### Instruction:\n{sample['instruction']}\n\n"
            f"### Response:\n{sample['output']}")

# ── Dataset ─────────────────────────────────────────────────────────

class LLMDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length):
        self.encodings = []
        for sample in samples:
            enc = tokenizer(
                sample["text"],
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )
            enc["labels"] = enc["input_ids"].clone()
            self.encodings.append({k: v.squeeze(0) for k, v in enc.items()})

    def __len__(self): return len(self.encodings)
    def __getitem__(self, idx): return self.encodings[idx]

def load_alpaca(num_samples, tokenizer, max_length):
    print(f"  Loading Alpaca ({num_samples} samples)...")
    ds = load_dataset("tatsu-lab/alpaca", split="train")
    ds = ds.select(range(min(num_samples, len(ds))))
    samples = [{"text": format_instruction(s)} for s in ds]
    return LLMDataset(samples, tokenizer, max_length)

# ── Model loader ────────────────────────────────────────────────────

def load_model_and_tokenizer(config: ExperimentConfig):
    print(f"  Loading tokenizer: {config.model}")
    tokenizer = AutoTokenizer.from_pretrained(config.model)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    if config.method == "qlora":
        from transformers import BitsAndBytesConfig
        print("  Loading model in 4-bit (QLoRA)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            config.model,
            quantization_config=bnb_config,
            device_map="auto"
        )
        model = prepare_model_for_kbit_training(model)

    elif config.method in ["lora", "full"]:
        print(f"  Loading model in fp16 ({config.method})...")
        model = AutoModelForCausalLM.from_pretrained(
            config.model,
            torch_dtype=torch.float16,
            device_map="auto"
        )

    return model, tokenizer

# ── PEFT / LoRA setup ───────────────────────────────────────────────

def apply_lora(model, config: ExperimentConfig):
    lora_config = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=config.target_modules,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

# ── Training stats tracker ──────────────────────────────────────────

class StatsCallback:
    def __init__(self):
        self.loss_history = []
        self.start_time = None

    def on_train_begin(self):
        self.start_time = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def on_log(self, logs):
        if "loss" in logs:
            self.loss_history.append(round(logs["loss"], 4))

    def get_stats(self):
        elapsed = round(time.time() - self.start_time, 1)
        peak_mem = 0
        if torch.cuda.is_available():
            peak_mem = round(
                torch.cuda.max_memory_allocated() / 1024**3, 2
            )
        return {
            "train_time_seconds": elapsed,
            "peak_gpu_memory_gb": peak_mem,
            "loss_history": self.loss_history
        }

# ── Main trainer ────────────────────────────────────────────────────

def run_experiment(config: ExperimentConfig, drive_base: str = "/content/drive/MyDrive/llm-lab"):
    print(f"\n{'='*55}")
    print(f"Running: {config.name}")
    print(f"Method: {config.method.upper()} | Rank: {config.lora_rank} | Dataset: {config.dataset}")
    print(f"{'='*55}")

    # 1. Load model
    model, tokenizer = load_model_and_tokenizer(config)

    # 2. Apply LoRA if needed
    if config.method in ["lora", "qlora"]:
        model = apply_lora(model, config)

    # Count trainable params
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    # 3. Load dataset
    dataset = load_alpaca(config.num_samples, tokenizer, config.max_length)

    # 4. Training arguments
    output_path = os.path.join(drive_base, "models", config.name)
    os.makedirs(output_path, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_path,
        num_train_epochs=config.epochs,
        per_device_train_batch_size=config.batch_size,
        gradient_accumulation_steps=4,
        learning_rate=config.learning_rate,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",       # no wandb for now
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
    )

    # 5. Trainer
    stats = StatsCallback()
    stats.on_train_begin()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    # 6. Train
    print("\n  Starting training...")
    result = trainer.train()
    stats.on_log({"loss": result.training_loss})

    # 7. Save adapter weights
    print("\n  Saving model...")
    if config.method in ["lora", "qlora"]:
        model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)

    # 8. Save metadata
    training_stats = stats.get_stats()
    metadata = {
        "name": config.name,
        "model": config.model,
        "method": config.method,
        "lora_rank": config.lora_rank,
        "dataset": config.dataset,
        "num_samples": config.num_samples,
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "learning_rate": config.learning_rate,
        "trainable_params": trainable,
        "total_params": total,
        **training_stats
    }

    meta_path = os.path.join(output_path, "training_metadata.json")
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\n  Done.")
    print(f"  Train time : {training_stats['train_time_seconds']}s")
    print(f"  Peak GPU   : {training_stats['peak_gpu_memory_gb']} GB")
    print(f"  Saved to   : {output_path}")

    return metadata

In [5]:
config_r4 = ExperimentConfig(
    name="tinyllama_lora_r4_alpaca",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    method="lora",
    lora_rank=4,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    dataset="alpaca",
    task="instruction",
    num_samples=200,
    epochs=1,
    batch_size=4,
    learning_rate=2e-4,
    max_length=512,
    output_dir="models/tinyllama_lora_r4_alpaca"
)

metadata_r4 = run_experiment(config_r4)


Running: tinyllama_lora_r4_alpaca
Method: LORA | Rank: 4 | Dataset: alpaca
  Loading tokenizer: TinyLlama/TinyLlama-1.1B-Chat-v1.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



  Starting training...


Step,Training Loss
10,1.807533



  Saving model...

  Done.
  Train time : 33.0s
  Peak GPU   : 6.16 GB
  Saved to   : /content/drive/MyDrive/llm-lab/models/tinyllama_lora_r4_alpaca
